# MiniAI — entraînement sur GPU (Google Colab)

Avant de lancer : **Exécution → Modifier le type d'exécution → GPU (T4)**.

Remplacez `VOTRE_COMPTE` par votre identifiant GitHub dans la cellule 1.

In [ ]:
# 1. Cloner le projet et installer les dépendances (torch est déjà installé sur Colab)
!git clone https://github.com/VOTRE_COMPTE/my-model.git
%cd my-model
!pip install -q datasets
import torch; print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN — changez le type d\'exécution')

In [ ]:
# 2. Corpus « Chinchilla » pour 19,4 M de paramètres : 20 tokens/param ≈ 390 M tokens ≈ 1,3 G caractères
#    (≈ 15 min de téléchargement + ≈ 20 min de préparation). Pour un test rapide : --max-chars 20000000.
!python main.py download-data --preset wiki-fr --max-chars 1300000000
!python main.py prepare-data

In [ ]:
# 3. Entraînement Chinchilla : 24 000 steps × 64 × 256 = 390 M tokens = 1 passage sur le corpus (~2 h 30 sur T4)
#    En cas de coupure de session : relancez avec --resume checkpoints/last.pt (même --max-steps).
!python main.py train --preset large --device cuda --max-steps 24000 --batch-size 64 --warmup 500 \
    --eval-interval 500 --run-name large-chinchilla --sample "Il est né à"

In [ ]:
# 4. Suivre l'entraînement : lancez cette cellule dans un second onglet pendant que la 3 tourne
import json
for line in open('checkpoints/train_log.jsonl', encoding='utf-8'):
    r = json.loads(line)
    if r['type'] == 'eval':
        print(f"step {r['step']:>6}  train {r['train_loss']:.3f}  val {r['val_loss']:.3f}  ppl {r['val_perplexity']:.1f}  acc {r['val_accuracy']*100:.1f}%  sondes {(r['probe_score'] or 0)*100:.0f}%")

In [ ]:
# 5. Tester
!python main.py generate --prompt "Il est né à" --max-tokens 80 -n 3 --temperature 0.7 --repetition-penalty 1.2

In [ ]:
# 6. Sauvegarder le résultat sur Google Drive (best.pt + tokenizer.json + log vont ensemble)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/miniai
!cp checkpoints/best.pt checkpoints/train_log.jsonl data/processed/tokenizer.json /content/drive/MyDrive/miniai/
print('Copié dans Drive/miniai — remettez best.pt dans checkpoints/ et tokenizer.json dans data/processed/ sur votre PC')